In [36]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.model_selection import KFold, cross_val_score


In [3]:
df = pd.read_csv('C:/WORKING_FOLDER/PY_narxoz/MACHINE_LEARNING/FILES/cars_fuel_efficiency.csv')

observe

In [4]:
print(df.shape)
print(df.dtypes)
print(df.describe())

(398, 9)
model                        object
year                          int64
cylinders                     int64
engine_litres               float64
power_hp                    float64
mass_kg                     float64
accel_0_100_s               float64
region                       object
fuel_efficiency_km_per_l    float64
dtype: object
              year   cylinders  engine_litres    power_hp      mass_kg  \
count   398.000000  398.000000     398.000000  392.000000   398.000000   
mean   1976.010050    5.454774       3.169698  104.469388  1347.366834   
std       3.697627    1.701004       1.708993   38.491160   384.123755   
min    1970.000000    3.000000       1.110000   46.000000   732.000000   
25%    1973.000000    4.000000       1.705000   75.000000  1008.500000   
50%    1976.000000    4.000000       2.430000   93.500000  1271.500000   
75%    1979.000000    8.000000       4.290000  126.000000  1636.500000   
max    1982.000000    8.000000       7.460000  230.000000  23

In [5]:
print(df.isna().sum())

model                       0
year                        0
cylinders                   0
engine_litres               0
power_hp                    6
mass_kg                     0
accel_0_100_s               0
region                      0
fuel_efficiency_km_per_l    0
dtype: int64


In [17]:
df = df.dropna(subset = ['power_hp', 'fuel_efficiency_km_per_l'])

linear regression

In [7]:
df.corr(method='pearson', min_periods=1, numeric_only=True)

,year,cylinders,engine_litres,power_hp,mass_kg,accel_0_100_s,fuel_efficiency_km_per_l
year,1.000000,-0.348746,-0.370035,-0.416361,-0.306539,0.288137,0.579351
cylinders,-0.348746,1.000000,0.950743,0.842983,0.895995,-0.505419,-0.775316
engine_litres,-0.370035,0.950743,1.000000,0.897197,0.932827,-0.543520,-0.804224
power_hp,-0.416361,0.842983,0.897197,1.000000,0.864540,-0.689196,-0.778400
mass_kg,-0.306539,0.895995,0.932827,0.864540,1.000000,-0.417451,-0.831690
accel_0_100_s,0.288137,-0.505419,-0.543520,-0.689196,-0.417451,1.000000,0.420284
fuel_efficiency_km_per_l,0.579351,-0.775316,-0.804224,-0.778400,-0.831690,0.420284,1.000000


In [ ]:
model = LinearRegression()
X = df[['engine_litres', 'power_hp']]
Y = df['fuel_efficiency_km_per_l']

X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=2463
)

model.fit(X_train, y_train)
print(f'Intercept: {model.intercept_:.2f} \nCoefficient: {model.coef_:.2f}')

Intercept: 16.12139650967748 
Coefficient: [-0.96081807 -0.03008804]


Report MAE, MSE, RMSE, and R² separately for the training set and the test set. Then repeat with 5-fold cross-validation (sklearn.model_selection.KFold or cross_val_score) and report the mean and standard deviation of the test-fold R².

In [30]:
y_train_predict = model.predict(X_train)
y_test_predict = model.predict(X_test)

In [35]:
def prediction(y_true, y_pred, name): 
    mae = mean_absolute_error(y_true, y_pred)
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_true, y_pred)
    print(f"{name:5s} ::: MAE={mae:.4f} : MSE={mse:.4f} : RMSE={rmse:.4f} : R^2={r2:.4f}")
    
prediction(y_train, y_train_predict, 'train')
prediction(y_test, y_test_predict, 'test')

train ::: MAE=1.4668 : MSE=3.6269 : RMSE=1.9045 : R^2=0.6645
test  ::: MAE=1.5099 : MSE=3.9729 : RMSE=1.9932 : R^2=0.6579


r squared approximately the same, so its appropriate

In [39]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(LinearRegression(), X, Y, cv = kf, scoring='r2')

print("R^2 by folds:", np.round(scores, 4))
print(f"Mean R2 = {scores.mean():.4f}")
print(f"STD R^2 = {scores.std():.4f}")

R^2 by folds: [0.5988 0.5783 0.6719 0.7268 0.6719]
Mean R2 = 0.6495
STD R^2 = 0.0541


Starting from your best single feature, add the remaining numeric features one group at a time and re-run Step 4's 5-fold cross-validation after each addition. Fill in a table like this one (rows are yours to choose; the shape matters, not these exact groupings): 
Answer: did test R² improve every time you added a feature? If not, which addition didn't help, and what does that tell you about throwing every column into the model? (This is the same question Lecture 4 asks in its Q&A slides on multiple linear regression — answer it from your own numbers, not from memory of the slide.)

In [41]:
df.columns

Index(['model', 'year', 'cylinders', 'engine_litres', 'power_hp', 'mass_kg',
       'accel_0_100_s', 'region', 'fuel_efficiency_km_per_l'],
      dtype='object')

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
target = 'fuel_efficiency_km_per_l'

numeric_cols = df.select_dtypes(include = 'number').columns.drop(target).tolist()
data = df[numeric_cols + [target]].dropna()

def for_mean(columns):
    scores = cross_val_score(LinearRegression(), df[columns], df[target], cv = kf, scoring='r2')
    return scores.mean(), scores.std()

single = {c: for_mean([c])[0] for c in numeric_cols}

for c, r2 in sorted(single.items(), key=lambda x: -x[1]):
    print(f"{c:20s} R2 = {r2:.4f}")
best = max(single, key=single.get)

mass_kg              R2 = 0.6852
engine_litres        R2 = 0.6374
cylinders            R2 = 0.5934
power_hp             R2 = 0.5923
year                 R2 = 0.3223
accel_0_100_s        R2 = 0.1547
